In [ ]:
%cd ../

In [ ]:
from app.imc2025.prediction import load_from_train


samples = load_from_train("./data")

In [ ]:
from mts.helpers.project.project import Project


last_project_iteration = Project.from_next_iteration("iterations")

In [ ]:
dataset_name = "imc2023_haiper"

In [ ]:
repositories_dirpath = last_project_iteration.iteration_dirpath / "h5_repositories"

In [ ]:
repositories_dirpath.mkdir(exist_ok=True, parents=True)

In [ ]:
from mts.pipeline.repository import h5 as h5_repo

In [ ]:
image_repository = h5_repo.H5ImageRepository.from_filename(
    repositories_dirpath, dataset_name
)
image_repository.add_repository_metadata(dataset_name=dataset_name)

In [ ]:
for prediction in samples["imc2023_haiper"]:
    image_repository.add_image(prediction.image_filepath)

In [ ]:
local_model_directory = (
    "checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"
)
retrival_model_dir = "checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric_retrieval_trainingfree.pth"
scene_graph = "retrieval-20-40"

In [ ]:
from mts.pipeline.step.pair.mast3r import Mast3rDistanceParer

distance_parer = Mast3rDistanceParer.from_checkpoints(
    local_model_directory,
    retrival_model_dir,
)

In [ ]:
from mts.pipeline.step.match.mast3r.merged import Mast3rMatchPipelineStep


matcher = Mast3rMatchPipelineStep.from_checkpoint(local_model_directory)

In [ ]:
pipeline_state = {}

In [ ]:
import torch

In [ ]:
cuda_device = torch.device("cuda:0")

In [ ]:
distance_parer.to(cuda_device)

In [ ]:
from config.logging import setup, DEBUG

setup(DEBUG)

In [ ]:
distance_parer.run(
    image_repository=image_repository,
    state=pipeline_state,
)

In [ ]:
matcher.to(cuda_device)

In [ ]:
matcher.run(
    image_repository=image_repository,
    state=pipeline_state,
)
